<div dir="rtl">

# الدرس 50: طرق توليد النص

دفتر الكود لهذا الدرس. شغّل الخلايا بالترتيب من الأعلى.

</div>

<div dir="rtl">

### دالة الاختيار

</div>

In [ ]:
import os
import re
import torch

def next_token_probs(logits, temperature=1.0, top_k=None, top_p=None):
    """Turn raw scores into the final sampling distribution."""
    if temperature == 0:                                   # greedy
        p = torch.zeros_like(logits)
        p[logits.argmax()] = 1.0
        return p
    p = torch.softmax(logits / temperature, dim=-1)
    if top_k is not None:
        kth = torch.topk(p, top_k).values[-1]
        p = torch.where(p >= kth, p, torch.zeros_like(p))
    if top_p is not None:
        sorted_p, order = torch.sort(p, descending=True)
        mass_before = torch.cumsum(sorted_p, 0) - sorted_p
        sorted_p[mass_before >= top_p] = 0                 # the set already reached top_p
        p = torch.zeros_like(p).scatter(0, order, sorted_p)
    return p / p.sum()

z = torch.tensor([2.0, 1.0, 0.0, -1.0])
for name, kw in [("T = 1", {}), ("T = 0.5", dict(temperature=0.5)), ("T = 2", dict(temperature=2.0)),
                 ("top-k = 2", dict(top_k=2)), ("top-p = 0.9", dict(top_p=0.9)), ("greedy", dict(temperature=0))]:
    print(f"{name:12}", next_token_probs(z, **kw).numpy().round(3))

<div dir="rtl">

### نموذجنا من الدرس 49

</div>

In [ ]:
import math
from dataclasses import dataclass
import torch
import torch.nn.functional as F
from torch import nn

@dataclass
class GPTConfig:
    vocab_size: int = 50257
    block_size: int = 1024          # context window
    n_layer: int = 12
    n_head: int = 12
    d_model: int = 768
    dropout: float = 0.0

class CausalSelfAttention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.n_head = cfg.n_head
        self.qkv = nn.Linear(cfg.d_model, 3 * cfg.d_model)
        self.proj = nn.Linear(cfg.d_model, cfg.d_model)
        self.dropout = cfg.dropout

    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=2)
        q, k, v = (t.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) for t in (q, k, v))
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True,
                                           dropout_p=self.dropout if self.training else 0.0)
        return self.proj(y.transpose(1, 2).reshape(B, T, C))

class MLP(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.fc = nn.Linear(cfg.d_model, 4 * cfg.d_model)
        self.proj = nn.Linear(4 * cfg.d_model, cfg.d_model)
        self.drop = nn.Dropout(cfg.dropout)

    def forward(self, x):
        return self.drop(self.proj(F.gelu(self.fc(x))))

class Block(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.ln1 = nn.LayerNorm(cfg.d_model)
        self.attn = CausalSelfAttention(cfg)
        self.ln2 = nn.LayerNorm(cfg.d_model)
        self.mlp = MLP(cfg)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))      # tokens communicate
        x = x + self.mlp(self.ln2(x))       # each token computes
        return x

class GPT(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.pos_emb = nn.Embedding(cfg.block_size, cfg.d_model)
        self.drop = nn.Dropout(cfg.dropout)
        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layer)])
        self.ln_f = nn.LayerNorm(cfg.d_model)
        self.head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight            # weight tying
        self.apply(self._init_weights)

    def _init_weights(self, m):                            # GPT-2 style initialization
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
        if isinstance(m, nn.Linear) and m.bias is not None:
            nn.init.zeros_(m.bias)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        assert T <= self.cfg.block_size, "sequence longer than the context window"
        pos = torch.arange(T, device=idx.device)
        x = self.drop(self.tok_emb(idx) + self.pos_emb(pos))
        for block in self.blocks:
            x = block(x)
        logits = self.head(self.ln_f(x))                   # (B, T, vocab_size)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

In [ ]:
if not os.path.exists("tiny_gpt.pt"):
    raise FileNotFoundError("Run the notebook of lesson 49 first: it trains and saves tiny_gpt.pt")
ckpt = torch.load("tiny_gpt.pt")
cfg = GPTConfig(**ckpt["config"])
model = GPT(cfg)
model.load_state_dict(ckpt["model"])
model.eval()
chars = ckpt["chars"]
stoi = {c: i for i, c in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]
decode = lambda ids: "".join(chars[i] for i in ids)
print("loaded:", sum(p.numel() for p in model.parameters()), "parameters, context", cfg.block_size)

@torch.no_grad()
def generate(prompt, n_new, seed=0, **decoding):
    g = torch.Generator().manual_seed(seed)
    idx = encode(prompt)
    for _ in range(n_new):
        logits, _ = model(torch.tensor([idx[-cfg.block_size:]]))
        p = next_token_probs(logits[0, -1], **decoding)
        idx.append(torch.multinomial(p, 1, generator=g).item())
    return decode(idx)

<div dir="rtl">

### ماذا يتوقع النموذج هنا؟

</div>

In [ ]:
@torch.no_grad()
def top_next(context, k=5):
    logits, _ = model(torch.tensor([encode(context)[-cfg.block_size:]]))
    p = torch.softmax(logits[0, -1], -1)
    top = torch.topk(p, k)
    return [(repr(chars[i]), round(v.item(), 3)) for v, i in zip(top.values, top.indices)]

print(top_next("KING RICHARD III:\nWhat say you, my lo"))
print(top_next("ROMEO:\n"))

<div dir="rtl">

### الجشع يعلق

</div>

In [ ]:
print(generate("ROMEO:\n", 300, temperature=0))

<div dir="rtl">

### أربع طرق، نفس البداية

</div>

In [ ]:
settings = {
    "temperature 0.7": dict(temperature=0.7),
    "temperature 1.3": dict(temperature=1.3),
    "top-k 5": dict(top_k=5),
    "top-p 0.9": dict(top_p=0.9),
}
for name, kw in settings.items():
    print(f"--- {name}")
    print(generate("JULIET:\n", 160, seed=3, **kw))

<div dir="rtl">

### قياس المقايضة

</div>

In [ ]:
import urllib.request
URL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
if not os.path.exists("tinyshakespeare.txt"):
    urllib.request.urlretrieve(URL, "tinyshakespeare.txt")
text = open("tinyshakespeare.txt", encoding="utf-8").read()
known = set(re.findall(r"[a-z']+", text.lower()))

def measure(sample):
    words = re.findall(r"[a-z']+", sample.lower())
    real = sum(w in known for w in words) / len(words)
    distinct = len(set(words)) / len(words)
    return real, distinct

for name, kw in {"greedy": dict(temperature=0), **settings, "pure sampling": {}}.items():
    real, distinct = measure(generate("\n", 800, seed=5, **kw))
    print(f"{name:16} real words {real:.0%} | distinct words {distinct:.0%}")

<div dir="rtl">

## تمرين: احسب، ثم جرّب

الدرجات: $(3, 1, 0)$.

1. احسب بيدك التوزيع بحرارة 1.
2. ما الكلمات الباقية مع أفضل نسبة 0.9؟ وما التوزيع بعد إعادة التقسيم؟
3. تحقق بدالة الاختيار، ثم ولّد من نموذجك بأفضل نسبة 0.5، ثم 0.95، وقارن.

</div>

In [ ]:
# اكتب حلّك هنا
